# P11 - Indian food recognition with CLIP (Kaggle runner)
Settings: Accelerator **GPU T4**, Internet **On**. Add input dataset `iamsouravbanerjee/indian-food-images-dataset`.
Run the setup cells, then **only your work package section**. All logic lives in `scripts/`; do not add logic here.

Each step below only calls a script in `scripts/`, passing `$SET` so paths and values come from `configs/default.yaml`.

## Setup

In [ ]:
# Clone the repo, install the two packages Kaggle lacks (pinned versions), then run the tests as a sanity check.
# Change BRANCH to test a feature branch.
REPO_URL = 'https://github.com/manishtiwari2/indian-food-clip.git'
BRANCH = 'main'
!git clone -b $BRANCH $REPO_URL repo
%cd repo
!pip install -q open_clip_torch==2.26.1 imagehash==4.3.1
!python -m pytest -q

In [ ]:
# Find the attached Indian food dataset under /kaggle/input and build the --set override string.
# SET points the scripts at that dataset and at a writable Food-101 folder; later cells reuse it.
import glob, os
cands = glob.glob('/kaggle/input/*indian*food*')
assert cands, 'add the Indian Food Images dataset as input'
INDIAN = cands[0]
SET = f'paths.indian_raw={INDIAN} paths.food101_root=/kaggle/working/food101'
print(SET)

## WP1 - data (run once; commit `splits/*.csv` via PR after lead's OK)

In [ ]:
# Clean, dedup and split the Indian data and prepare the Food-101 subset (downloaded to /kaggle/working).
# Run once; the lead commits the resulting splits/*.csv.
!python scripts/prepare_data.py --food101 --download-food101 --set $SET

## Food-101 images (WP3/WP4 need them if splits are already committed)

In [ ]:
# Only fetch Food-101 images (skip the Indian step) because splits are already committed.
!python scripts/prepare_data.py --skip-indian --download-food101 --set $SET

## WP2 - CNN baseline

In [ ]:
# E1: train the from-scratch CNN baseline; --run-name labels this run.
!python scripts/train_cnn.py --run-name baseline --set $SET

## WP3 - CLIP features + zero-shot (save `features/` as a Kaggle dataset for WP4)

In [ ]:
# Cache CLIP image features for the models in the config, then run zero-shot (E2-E4) on the cached features.
!python scripts/extract_features.py --set $SET
!python scripts/run_zeroshot.py --set $SET

## WP4 - few-shot + figures

In [ ]:
# E5: linear-probe few-shot runs, then build figures and the E6 gap table from results/*.csv.
!python scripts/run_fewshot.py --set $SET
!python scripts/make_figures.py --set $SET

## D13 fine-tune (ViT-B/16, about 10 min on a T4)

In [ ]:
# E7: fine-tune the CLIP image encoder (settings in the finetune block of configs/default.yaml).
!python scripts/finetune_clip.py --set $SET

## Download outputs (commit `results/` and `report/figures/` in your PR)

In [ ]:
# Bundle results, figures and splits into one zip to download, and list the files as a check.
!zip -r /kaggle/working/outputs.zip results report/figures splits
!ls -la results report/figures